Review Project Setup and Data Generaion
## FPL Revenue Collections Risk Analytics
Purpose: Setup the environment, load raw datasets, and generate a synthetic utility customer dataset that mirrors collection workflow and NextEra Energy

In [5]:
import importlib
import sys

required = [
    "pandas", "numpy", "matplotlib", "seaborn",
    "sklearn", "imblearn", "scipy", "openpyxl"
]

print(f"Python version: {sys.version.split()[0]}\n")
print("Library              Status")
print("-" * 35)

for lib in required:
    try:
        mod = importlib.import_module(lib)
        ver = str(getattr(mod, "__version__", "ok"))
        print(f"  {lib.ljust(18)} OK  {ver}")
    except ImportError:
        print(f"  {lib.ljust(18)} NOT INSTALLED")

Python version: 3.9.13

Library              Status
-----------------------------------
  pandas             OK  2.2.2
  numpy              OK  1.26.4
  matplotlib         OK  3.8.4
  seaborn            OK  0.13.2
  sklearn            OK  1.4.2
  imblearn           OK  0.12.2
  scipy              OK  1.13.0
  openpyxl           OK  3.1.2


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings("ignore")  #shut up yellow warnings!!!
np.random.seed(42)

# Paths
RAW_DIR       = "../data/raw"
PROCESSED_DIR = "../data/processed"
FIGURES_DIR   = "../reports/figures"

# Directories to create
for path in [RAW_DIR, PROCESSED_DIR, FIGURES_DIR]:
    os.makedirs(path, exist_ok=True)

print("Libraries imported successfully.")

Libraries imported successfully.


## LOADING UCI DATASET

In [7]:
UCI_FILE = os.path.join(RAW_DIR, "UCI dataset.xls")

uci_df = pd.read_excel(UCI_FILE, header=1)

print(f"Dataset loaded: {uci_df.shape[0]} rows, {uci_df.shape[1]} columns")
print(f"Columns: {list(uci_df.columns)}")

Dataset loaded: 30000 rows, 25 columns
Columns: ['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']


In [8]:
uci_df.columns = uci_df.columns.str.strip().str.lower().str.replace( " ", "_")

uci_df.rename(columns={"default_payment_next_month": "delinquent"}, inplace=True)

print(f"Columns: {list(uci_df.columns)}")

Columns: ['id', 'limit_bal', 'sex', 'education', 'marriage', 'age', 'pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6', 'bill_amt1', 'bill_amt2', 'bill_amt3', 'bill_amt4', 'bill_amt5', 'bill_amt6', 'pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt4', 'pay_amt5', 'pay_amt6', 'delinquent']


In [20]:
print("=" * 45)
print(" UCI DATASET PROFILE")
print("=" * 45)
print(f" Rows          : {uci_df.shape[0]:,}")
print(f" Columns       : {uci_df.shape[1]}")
print(f" Null values   : {uci_df.isnull().sum().sum()}")
print(f" Duplicates    : {uci_df.duplicated().sum()}")
print("=" * 45)

#Delinquency rate
total = len(uci_df)
delinquent = uci_df["delinquent"].sum()

print(f"\n Delinquent accounts     : {delinquent:,}")
print(f"\n Non-Delinquent accounts : {total - delinquent:,}")
print(f"\n Delinquency Rate        : {delinquent/total*100:.1f}%")

uci_df.head()

 UCI DATASET PROFILE
 Rows          : 30,000
 Columns       : 25
 Null values   : 0
 Duplicates    : 0

 Delinquent accounts     : 6,636

 Non-Delinquent accounts : 23,364

 Delinquency Rate        : 22.1%


,id,limit_bal,sex,education,marriage,age,pay_0,pay_2,pay_3,pay_4,...,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,delinquent
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0
